# Análise exploratória | MRBR | vw_ds_pro_mrbr_bloq_pagamento_man

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`` |
| Ambiente | DEV |
| Objeto | View |
| Colunas do DESCRIBE | 32 |
| Tipos | decimal(18,3), int, string |
| Clustering declarado | Não informado no CSV |
| Centro | Não consta; `cod_empresa` é apenas recorte alternativo, NÃO centro |
| Chave candidata | `num_doc_faturamento` + `num_exercicio` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_MRBR_1.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | Sem cod_centro: não presumir equivalência do filtro SAP por centro. |
| 🔴 ALTA | 2 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |
| 🟡 MÉDIA | View: conferir comentário no DESCRIBE EXTENDED; DETAIL/HISTORY não executados. |
| 🟡 MÉDIA | Há campos documentados como sem fonte/indisponíveis; verificar preenchimento sem classificá-los automaticamente como erro. |
| 🟡 MÉDIA | Há datas STRING: medir formato e sentinelas antes de converter. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`;


### 1.1. DETAIL/HISTORY da view

Não executados: estes comandos não são válidos para view. Consultar o comentário da view na seção 1.


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `num_doc_faturamento`) AS distintos_num_doc_faturamento, COUNT(DISTINCT `cod_empresa`) AS cod_empresa_distintos_nao_centros FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`;


## 3. Distribuição por centro

O schema não contém cod_centro; a consulta abaixo mostra somente um recorte alternativo, sem equivalência ao centro SAP.


In [0]:
SELECT `cod_empresa` AS recorte_alternativo_nao_centro, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_empresa` ORDER BY linhas DESC LIMIT 100;


## 3.1. Centros 4128 e 4014

Não existe campo de centro confirmado. Esta célula registra a limitação sem aplicar filtro inventado.


In [0]:
SELECT 'SEM_COLUNA_CENTRO' AS situacao, '4128 e 4014 nao podem ser medidos diretamente neste schema' AS observacao;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_doc_faturamento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio + cod_empresa' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio`, `cod_empresa` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio + cod_empresa + cod_emissor_fatura' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio`, `cod_empresa`, `cod_emissor_fatura` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_doc_faturamento`, t.`num_exercicio`, md5(to_json(struct(t.`num_exercicio`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`sg_moeda`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_montante_1`, t.`ind_bloq_manual_1`, t.`num_linha_servico`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`dt_conversao`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` t), d AS (SELECT `num_doc_faturamento`, `num_exercicio`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_doc_faturamento`, `num_exercicio` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`num_exercicio` AS STRING)) AS `dist_num_exercicio`, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS `dist_cod_empresa`, COUNT(DISTINCT CAST(`num_doc_faturamento` AS STRING)) AS `dist_num_doc_faturamento`, COUNT(DISTINCT CAST(`cod_emissor_fatura` AS STRING)) AS `dist_cod_emissor_fatura`, COUNT(DISTINCT CAST(`dt_base_prazo_pgto` AS STRING)) AS `dist_dt_base_prazo_pgto`, COUNT(DISTINCT CAST(`dt_lancamento` AS STRING)) AS `dist_dt_lancamento`, COUNT(DISTINCT CAST(`qt_dias_desconto_1` AS STRING)) AS `dist_qt_dias_desconto_1`, COUNT(DISTINCT CAST(`sg_moeda` AS STRING)) AS `dist_sg_moeda`, COUNT(DISTINCT CAST(`sg_moeda_interna` AS STRING)) AS `dist_sg_moeda_interna`, COUNT(DISTINCT CAST(`tp_documento` AS STRING)) AS `dist_tp_documento`, COUNT(DISTINCT CAST(`cod_bloq_pagamento_log` AS STRING)) AS `dist_cod_bloq_pagamento_log`, COUNT(DISTINCT CAST(`cod_bloqueio_pgto` AS STRING)) AS `dist_cod_bloqueio_pgto` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `num_doc_faturamento`, `num_exercicio` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_num_exercicio` > 1) AS `chaves_variam_num_exercicio`, COUNT_IF(`dist_cod_empresa` > 1) AS `chaves_variam_cod_empresa`, COUNT_IF(`dist_num_doc_faturamento` > 1) AS `chaves_variam_num_doc_faturamento`, COUNT_IF(`dist_cod_emissor_fatura` > 1) AS `chaves_variam_cod_emissor_fatura`, COUNT_IF(`dist_dt_base_prazo_pgto` > 1) AS `chaves_variam_dt_base_prazo_pgto`, COUNT_IF(`dist_dt_lancamento` > 1) AS `chaves_variam_dt_lancamento`, COUNT_IF(`dist_qt_dias_desconto_1` > 1) AS `chaves_variam_qt_dias_desconto_1`, COUNT_IF(`dist_sg_moeda` > 1) AS `chaves_variam_sg_moeda`, COUNT_IF(`dist_sg_moeda_interna` > 1) AS `chaves_variam_sg_moeda_interna`, COUNT_IF(`dist_tp_documento` > 1) AS `chaves_variam_tp_documento`, COUNT_IF(`dist_cod_bloq_pagamento_log` > 1) AS `chaves_variam_cod_bloq_pagamento_log`, COUNT_IF(`dist_cod_bloqueio_pgto` > 1) AS `chaves_variam_cod_bloqueio_pgto` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(24, 'num_exercicio', 'string', c0_n, c0_v, c0_z, 'cod_empresa', 'string', c1_n, c1_v, c1_z, 'nm_fornecedor', 'string', c2_n, c2_v, c2_z, 'num_doc_faturamento', 'string', c3_n, c3_v, c3_z, 'cod_emissor_fatura', 'string', c4_n, c4_v, c4_z, 'dt_base_prazo_pgto', 'string', c5_n, c5_v, c5_z, 'dt_lancamento', 'string', c6_n, c6_v, c6_z, 'qt_dias_desconto_1', 'int', c7_n, c7_v, c7_z, 'sg_moeda', 'string', c8_n, c8_v, c8_z, 'sg_moeda_interna', 'string', c9_n, c9_v, c9_z, 'tp_documento', 'string', c10_n, c10_v, c10_z, 'cod_bloq_pagamento_log', 'string', c11_n, c11_v, c11_z, 'nm_usuario', 'string', c12_n, c12_v, c12_z, 'cod_bloqueio_pgto', 'string', c13_n, c13_v, c13_z, 'sg_upp', 'string', c14_n, c14_v, c14_z, 'pct_desconto_2', 'string', c15_n, c15_v, c15_z, 'pct_desconto_1', 'string', c16_n, c16_v, c16_z, 'vl_taxa_cambio', 'string', c17_n, c17_v, c17_z, 'st_fatura', 'string', c18_n, c18_v, c18_z, 'qt_em_upp', 'decimal(18,3)', c19_n, c19_v, c19_z, 'qt_diferenca', 'decimal(18,3)', c20_n, c20_v, c20_z, 'cod_determ_preco', 'string', c21_n, c21_v, c21_z, 'cod_pais_destino', 'string', c22_n, c22_v, c22_z, 'cod_pais_procedencia', 'string', c23_n, c23_v, c23_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_exercicio` IS NULL) AS c0_n, COUNT_IF(`num_exercicio` IS NOT NULL AND LOWER(TRIM(`num_exercicio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`cod_empresa` IS NULL) AS c1_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`nm_fornecedor` IS NULL) AS c2_n, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`num_doc_faturamento` IS NULL) AS c3_n, COUNT_IF(`num_doc_faturamento` IS NOT NULL AND LOWER(TRIM(`num_doc_faturamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`cod_emissor_fatura` IS NULL) AS c4_n, COUNT_IF(`cod_emissor_fatura` IS NOT NULL AND LOWER(TRIM(`cod_emissor_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS c5_n, COUNT_IF(`dt_base_prazo_pgto` IS NOT NULL AND LOWER(TRIM(`dt_base_prazo_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`dt_lancamento` IS NULL) AS c6_n, COUNT_IF(`dt_lancamento` IS NOT NULL AND LOWER(TRIM(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`qt_dias_desconto_1` IS NULL) AS c7_n, CAST(0 AS BIGINT) AS c7_v, COUNT_IF(`qt_dias_desconto_1` = 0) AS c7_z, COUNT_IF(`sg_moeda` IS NULL) AS c8_n, COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c8_v, COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS c8_z, COUNT_IF(`sg_moeda_interna` IS NULL) AS c9_n, COUNT_IF(`sg_moeda_interna` IS NOT NULL AND LOWER(TRIM(`sg_moeda_interna`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`sg_moeda_interna`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`tp_documento` IS NULL) AS c10_n, COUNT_IF(`tp_documento` IS NOT NULL AND LOWER(TRIM(`tp_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`tp_documento`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`cod_bloq_pagamento_log` IS NULL) AS c11_n, COUNT_IF(`cod_bloq_pagamento_log` IS NOT NULL AND LOWER(TRIM(`cod_bloq_pagamento_log`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`cod_bloq_pagamento_log`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`nm_usuario` IS NULL) AS c12_n, COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c12_v, COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS c12_z, COUNT_IF(`cod_bloqueio_pgto` IS NULL) AS c13_n, COUNT_IF(`cod_bloqueio_pgto` IS NOT NULL AND LOWER(TRIM(`cod_bloqueio_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c13_v, COUNT_IF(TRIM(`cod_bloqueio_pgto`) RLIKE '^0+([.,]0+)?$') AS c13_z, COUNT_IF(`sg_upp` IS NULL) AS c14_n, COUNT_IF(`sg_upp` IS NOT NULL AND LOWER(TRIM(`sg_upp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c14_v, COUNT_IF(TRIM(`sg_upp`) RLIKE '^0+([.,]0+)?$') AS c14_z, COUNT_IF(`pct_desconto_2` IS NULL) AS c15_n, COUNT_IF(`pct_desconto_2` IS NOT NULL AND LOWER(TRIM(`pct_desconto_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c15_v, COUNT_IF(TRIM(`pct_desconto_2`) RLIKE '^0+([.,]0+)?$') AS c15_z, COUNT_IF(`pct_desconto_1` IS NULL) AS c16_n, COUNT_IF(`pct_desconto_1` IS NOT NULL AND LOWER(TRIM(`pct_desconto_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c16_v, COUNT_IF(TRIM(`pct_desconto_1`) RLIKE '^0+([.,]0+)?$') AS c16_z, COUNT_IF(`vl_taxa_cambio` IS NULL) AS c17_n, COUNT_IF(`vl_taxa_cambio` IS NOT NULL AND LOWER(TRIM(`vl_taxa_cambio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c17_v, COUNT_IF(TRIM(`vl_taxa_cambio`) RLIKE '^0+([.,]0+)?$') AS c17_z, COUNT_IF(`st_fatura` IS NULL) AS c18_n, COUNT_IF(`st_fatura` IS NOT NULL AND LOWER(TRIM(`st_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`st_fatura`) RLIKE '^0+([.,]0+)?$') AS c18_z, COUNT_IF(`qt_em_upp` IS NULL) AS c19_n, CAST(0 AS BIGINT) AS c19_v, COUNT_IF(`qt_em_upp` = 0) AS c19_z, COUNT_IF(`qt_diferenca` IS NULL) AS c20_n, CAST(0 AS BIGINT) AS c20_v, COUNT_IF(`qt_diferenca` = 0) AS c20_z, COUNT_IF(`cod_determ_preco` IS NULL) AS c21_n, COUNT_IF(`cod_determ_preco` IS NOT NULL AND LOWER(TRIM(`cod_determ_preco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c21_v, COUNT_IF(TRIM(`cod_determ_preco`) RLIKE '^0+([.,]0+)?$') AS c21_z, COUNT_IF(`cod_pais_destino` IS NULL) AS c22_n, COUNT_IF(`cod_pais_destino` IS NOT NULL AND LOWER(TRIM(`cod_pais_destino`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c22_v, COUNT_IF(TRIM(`cod_pais_destino`) RLIKE '^0+([.,]0+)?$') AS c22_z, COUNT_IF(`cod_pais_procedencia` IS NULL) AS c23_n, COUNT_IF(`cod_pais_procedencia` IS NOT NULL AND LOWER(TRIM(`cod_pais_procedencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c23_v, COUNT_IF(TRIM(`cod_pais_procedencia`) RLIKE '^0+([.,]0+)?$') AS c23_z FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) a UNION ALL SELECT stack(8, 'ind_nota_existente', 'string', c24_n, c24_v, c24_z, 'num_pacote', 'string', c25_n, c25_v, c25_z, 'ind_bloq_montante_1', 'string', c26_n, c26_v, c26_z, 'ind_bloq_manual_1', 'string', c27_n, c27_v, c27_z, 'num_linha_servico', 'string', c28_n, c28_v, c28_z, 'qt_dias_liquido', 'int', c29_n, c29_v, c29_z, 'qt_dias_desconto_2', 'int', c30_n, c30_v, c30_z, 'dt_conversao', 'string', c31_n, c31_v, c31_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`ind_nota_existente` IS NULL) AS c24_n, COUNT_IF(`ind_nota_existente` IS NOT NULL AND LOWER(TRIM(`ind_nota_existente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c24_v, COUNT_IF(TRIM(`ind_nota_existente`) RLIKE '^0+([.,]0+)?$') AS c24_z, COUNT_IF(`num_pacote` IS NULL) AS c25_n, COUNT_IF(`num_pacote` IS NOT NULL AND LOWER(TRIM(`num_pacote`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c25_v, COUNT_IF(TRIM(`num_pacote`) RLIKE '^0+([.,]0+)?$') AS c25_z, COUNT_IF(`ind_bloq_montante_1` IS NULL) AS c26_n, COUNT_IF(`ind_bloq_montante_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c26_v, COUNT_IF(TRIM(`ind_bloq_montante_1`) RLIKE '^0+([.,]0+)?$') AS c26_z, COUNT_IF(`ind_bloq_manual_1` IS NULL) AS c27_n, COUNT_IF(`ind_bloq_manual_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c27_v, COUNT_IF(TRIM(`ind_bloq_manual_1`) RLIKE '^0+([.,]0+)?$') AS c27_z, COUNT_IF(`num_linha_servico` IS NULL) AS c28_n, COUNT_IF(`num_linha_servico` IS NOT NULL AND LOWER(TRIM(`num_linha_servico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c28_v, COUNT_IF(TRIM(`num_linha_servico`) RLIKE '^0+([.,]0+)?$') AS c28_z, COUNT_IF(`qt_dias_liquido` IS NULL) AS c29_n, CAST(0 AS BIGINT) AS c29_v, COUNT_IF(`qt_dias_liquido` = 0) AS c29_z, COUNT_IF(`qt_dias_desconto_2` IS NULL) AS c30_n, CAST(0 AS BIGINT) AS c30_v, COUNT_IF(`qt_dias_desconto_2` = 0) AS c30_z, COUNT_IF(`dt_conversao` IS NULL) AS c31_n, COUNT_IF(`dt_conversao` IS NOT NULL AND LOWER(TRIM(`dt_conversao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c31_v, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^0+([.,]0+)?$') AS c31_z FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte ou sem preenchimento declarado

Confere o comportamento documentado; 100% nulo não é automaticamente erro de carga.


In [0]:
WITH a AS (SELECT 'st_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`st_fatura` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'qt_diferenca' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_determ_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_determ_preco` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'ind_nota_existente' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_nota_existente` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'num_pacote' AS coluna, COUNT(*) AS total, COUNT_IF(`num_pacote` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'num_linha_servico' AS coluna, COUNT(*) AS total, COUNT_IF(`num_linha_servico` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) SELECT coluna, total, nulos, CASE WHEN total = nulos THEN 'CONFIRMADO NO RECORTE' ELSE 'HA PREENCHIMENTO: INVESTIGAR' END AS leitura FROM a ORDER BY coluna;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `num_doc_faturamento`) AS d0, COUNT(DISTINCT `cod_empresa`) AS d1, COUNT(DISTINCT `cod_emissor_fatura`) AS d2, COUNT(DISTINCT `sg_moeda`) AS d3, COUNT(DISTINCT `sg_moeda_interna`) AS d4, COUNT(DISTINCT `tp_documento`) AS d5, COUNT(DISTINCT `cod_bloq_pagamento_log`) AS d6 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(7, 'num_doc_faturamento', d0, 'cod_empresa', d1, 'cod_emissor_fatura', d2, 'sg_moeda', d3, 'sg_moeda_interna', d4, 'tp_documento', d5, 'cod_bloq_pagamento_log', d6) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_empresa` UNION ALL SELECT 'cod_emissor_fatura' AS coluna, CAST(`cod_emissor_fatura` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_emissor_fatura` UNION ALL SELECT 'sg_moeda' AS coluna, CAST(`sg_moeda` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `sg_moeda` UNION ALL SELECT 'sg_moeda_interna' AS coluna, CAST(`sg_moeda_interna` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `sg_moeda_interna` UNION ALL SELECT 'tp_documento' AS coluna, CAST(`tp_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `tp_documento` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, CAST(`cod_bloq_pagamento_log` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_bloq_pagamento_log` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, CAST(`cod_bloqueio_pgto` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_bloqueio_pgto` UNION ALL SELECT 'sg_upp' AS coluna, CAST(`sg_upp` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `sg_upp`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'tp_documento' AS coluna, CAST(`tp_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `tp_documento` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, CAST(`cod_bloq_pagamento_log` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_bloq_pagamento_log` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, CAST(`cod_bloqueio_pgto` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_bloqueio_pgto` UNION ALL SELECT 'st_fatura' AS coluna, CAST(`st_fatura` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `st_fatura` UNION ALL SELECT 'ind_nota_existente' AS coluna, CAST(`ind_nota_existente` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `ind_nota_existente` UNION ALL SELECT 'ind_bloq_montante_1' AS coluna, CAST(`ind_bloq_montante_1` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `ind_bloq_montante_1` UNION ALL SELECT 'ind_bloq_manual_1' AS coluna, CAST(`ind_bloq_manual_1` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `ind_bloq_manual_1`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(5, 'qt_dias_desconto_1', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero, 'qt_em_upp', CAST(n1_min AS DOUBLE), CAST(n1_max AS DOUBLE), CAST(n1_avg AS DOUBLE), CAST(n1_med AS DOUBLE), n1_neg, n1_zero, 'qt_diferenca', CAST(n2_min AS DOUBLE), CAST(n2_max AS DOUBLE), CAST(n2_avg AS DOUBLE), CAST(n2_med AS DOUBLE), n2_neg, n2_zero, 'qt_dias_liquido', CAST(n3_min AS DOUBLE), CAST(n3_max AS DOUBLE), CAST(n3_avg AS DOUBLE), CAST(n3_med AS DOUBLE), n3_neg, n3_zero, 'qt_dias_desconto_2', CAST(n4_min AS DOUBLE), CAST(n4_max AS DOUBLE), CAST(n4_avg AS DOUBLE), CAST(n4_med AS DOUBLE), n4_neg, n4_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`qt_dias_desconto_1`) AS n0_min, MAX(`qt_dias_desconto_1`) AS n0_max, AVG(`qt_dias_desconto_1`) AS n0_avg, PERCENTILE_APPROX(`qt_dias_desconto_1`, 0.5) AS n0_med, COUNT_IF(`qt_dias_desconto_1` < 0) AS n0_neg, COUNT_IF(`qt_dias_desconto_1` = 0) AS n0_zero, MIN(`qt_em_upp`) AS n1_min, MAX(`qt_em_upp`) AS n1_max, AVG(`qt_em_upp`) AS n1_avg, PERCENTILE_APPROX(`qt_em_upp`, 0.5) AS n1_med, COUNT_IF(`qt_em_upp` < 0) AS n1_neg, COUNT_IF(`qt_em_upp` = 0) AS n1_zero, MIN(`qt_diferenca`) AS n2_min, MAX(`qt_diferenca`) AS n2_max, AVG(`qt_diferenca`) AS n2_avg, PERCENTILE_APPROX(`qt_diferenca`, 0.5) AS n2_med, COUNT_IF(`qt_diferenca` < 0) AS n2_neg, COUNT_IF(`qt_diferenca` = 0) AS n2_zero, MIN(`qt_dias_liquido`) AS n3_min, MAX(`qt_dias_liquido`) AS n3_max, AVG(`qt_dias_liquido`) AS n3_avg, PERCENTILE_APPROX(`qt_dias_liquido`, 0.5) AS n3_med, COUNT_IF(`qt_dias_liquido` < 0) AS n3_neg, COUNT_IF(`qt_dias_liquido` = 0) AS n3_zero, MIN(`qt_dias_desconto_2`) AS n4_min, MAX(`qt_dias_desconto_2`) AS n4_max, AVG(`qt_dias_desconto_2`) AS n4_avg, PERCENTILE_APPROX(`qt_dias_desconto_2`, 0.5) AS n4_med, COUNT_IF(`qt_dias_desconto_2` < 0) AS n4_neg, COUNT_IF(`qt_dias_desconto_2` = 0) AS n4_zero FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(5, 'qt_dias_desconto_1', CAST(s0 AS DOUBLE), 'qt_em_upp', CAST(s1 AS DOUBLE), 'qt_diferenca', CAST(s2 AS DOUBLE), 'qt_dias_liquido', CAST(s3 AS DOUBLE), 'qt_dias_desconto_2', CAST(s4 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`qt_dias_desconto_1`) AS s0, SUM(`qt_em_upp`) AS s1, SUM(`qt_diferenca`) AS s2, SUM(`qt_dias_liquido`) AS s3, SUM(`qt_dias_desconto_2`) AS s4 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_base_prazo_pgto' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '') AS vazios, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_base_prazo_pgto`) AS menor_literal, MAX(`dt_base_prazo_pgto`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'dt_lancamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_lancamento` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_lancamento`) = '') AS vazios, COUNT_IF(TRIM(`dt_lancamento`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_lancamento`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_lancamento`) AS menor_literal, MAX(`dt_lancamento`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'dt_conversao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_conversao` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_conversao`) = '') AS vazios, COUNT_IF(TRIM(`dt_conversao`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_conversao`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_conversao`) AS menor_literal, MAX(`dt_conversao`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'num_exercicio' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_exercicio`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_exercicio`))) AS tamanho_max, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_exercicio`) RLIKE '^[0-9]+$' THEN TRIM(`num_exercicio`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_exercicio`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_exercicio`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_empresa`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_empresa`))) AS tamanho_max, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN TRIM(`cod_empresa`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_empresa`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'num_doc_faturamento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_doc_faturamento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_doc_faturamento`))) AS tamanho_max, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN TRIM(`num_doc_faturamento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_faturamento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_emissor_fatura' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_emissor_fatura`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_emissor_fatura`))) AS tamanho_max, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN TRIM(`cod_emissor_fatura`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_emissor_fatura`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_bloq_pagamento_log`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_bloq_pagamento_log`))) AS tamanho_max, COUNT_IF(TRIM(`cod_bloq_pagamento_log`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_bloq_pagamento_log`) RLIKE '^[0-9]+$' THEN TRIM(`cod_bloq_pagamento_log`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_bloq_pagamento_log`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_bloq_pagamento_log`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_bloqueio_pgto`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_bloqueio_pgto`))) AS tamanho_max, COUNT_IF(TRIM(`cod_bloqueio_pgto`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_bloqueio_pgto`) RLIKE '^[0-9]+$' THEN TRIM(`cod_bloqueio_pgto`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_bloqueio_pgto`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_bloqueio_pgto`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_determ_preco' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_determ_preco`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_determ_preco`))) AS tamanho_max, COUNT_IF(TRIM(`cod_determ_preco`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_determ_preco`) RLIKE '^[0-9]+$' THEN TRIM(`cod_determ_preco`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_determ_preco`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_determ_preco`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_pais_destino' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_pais_destino`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_pais_destino`))) AS tamanho_max, COUNT_IF(TRIM(`cod_pais_destino`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_pais_destino`) RLIKE '^[0-9]+$' THEN TRIM(`cod_pais_destino`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_pais_destino`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_pais_destino`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'cod_pais_procedencia' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_pais_procedencia`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_pais_procedencia`))) AS tamanho_max, COUNT_IF(TRIM(`cod_pais_procedencia`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_pais_procedencia`) RLIKE '^[0-9]+$' THEN TRIM(`cod_pais_procedencia`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_pais_procedencia`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_pais_procedencia`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` UNION ALL SELECT 'num_pacote' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_pacote`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_pacote`))) AS tamanho_max, COUNT_IF(TRIM(`num_pacote`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_pacote`) RLIKE '^[0-9]+$' THEN TRIM(`num_pacote`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_pacote`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_pacote`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`num_exercicio`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`sg_moeda`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_montante_1`, t.`ind_bloq_manual_1`, t.`num_linha_servico`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`dt_conversao`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_empresa`, `cod_emissor_fatura`, `sg_moeda`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_empresa`, `cod_emissor_fatura`, `sg_moeda` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Não há coluna de carga documentada; registrar a data e hora de execução da consulta.


In [0]:
SELECT CURRENT_TIMESTAMP() AS instante_da_consulta, 'Sem coluna de carga no schema' AS limitacao;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Bloqueios e datas em texto


In [0]:
SELECT `cod_empresa`, `cod_bloq_pagamento_log`, COUNT(*) AS linhas, COUNT_IF(`dt_lancamento` IS NOT NULL AND TRIM(`dt_lancamento`) <> '' AND TRY_TO_TIMESTAMP(`dt_lancamento`, 'dd/MM/yyyy') IS NULL) AS datas_lancamento_invalidas, COUNT_IF(`st_fatura` IS NOT NULL AND TRIM(`st_fatura`) <> '') AS status_fatura_preenchido FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `cod_empresa`, `cod_bloq_pagamento_log` ORDER BY linhas DESC LIMIT 100;


### 18.2. Cobertura cruzada das duas views MRBR

Compara apenas identificadores documentados em ambos os DESCRIBEs; a ausência de chave em uma view não define qual é oficial.


In [0]:
WITH a AS (SELECT num_exercicio, cod_empresa, num_doc_faturamento FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY num_exercicio, cod_empresa, num_doc_faturamento), b AS (SELECT num_exercicio, cod_empresa, num_doc_faturamento FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY num_exercicio, cod_empresa, num_doc_faturamento) SELECT COUNT(*) AS chaves_uniao, COUNT_IF(a.num_doc_faturamento IS NOT NULL AND b.num_doc_faturamento IS NOT NULL) AS em_ambas, COUNT_IF(a.num_doc_faturamento IS NOT NULL AND b.num_doc_faturamento IS NULL) AS apenas_mrbr_1, COUNT_IF(a.num_doc_faturamento IS NULL AND b.num_doc_faturamento IS NOT NULL) AS apenas_mrbr_2 FROM a FULL OUTER JOIN b ON a.num_exercicio <=> b.num_exercicio AND a.cod_empresa <=> b.cod_empresa AND a.num_doc_faturamento <=> b.num_doc_faturamento;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `num_exercicio`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `sg_moeda`, `sg_moeda_interna`, `tp_documento`, `cod_bloq_pagamento_log`, `cod_bloqueio_pgto`, `sg_upp`, `pct_desconto_2`, `pct_desconto_1`, `vl_taxa_cambio`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY `num_doc_faturamento`, `num_exercicio`), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man`) SELECT 'MRBR' AS transacao, 'vw_ds_pro_mrbr_bloq_pagamento_man' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, 'SEM CENTRO NO SCHEMA' AS filtro_centro, 'Chave candidata: num_doc_faturamento + num_exercicio' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
